# Import libraries

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.mllib.evaluation import BinaryClassificationMetrics
from pyspark.ml import Pipeline
from pyspark.sql.functions import expr

# Read historical customer churn data from silver catalog

In [2]:
training_data = spark.read.table("churn_silver.historical.customers")
# drop non signifcant column
training_data = training_data.drop("year", "sentimentScore_str")
training_data = training_data.dropna()

# Process categorical and numeric attributes.

In [3]:
categorical_cols = ["gender", "partner", "dependents", "phoneservice", "multiplelines", "internetservice", "onlinesecurity", "onlinebackup", "deviceprotection", "techsupport", "streamingtv", "streamingmovies", "contract", "paperlessbilling", "paymentmethod"]
indexers = [
    StringIndexer(inputCol=c, outputCol=c + "_idx", handleInvalid="skip")
    for c in categorical_cols
]
encoders = [OneHotEncoder(inputCol=c + "_idx", outputCol=c + "_vec") for c in categorical_cols]

# Numerical columns
numeric_cols = ["tenure", "monthlycharges", "totalcharges", "sentimentScore"]

# Assemble features
assembler = VectorAssembler(
    inputCols=[c + "_vec" for c in categorical_cols] + numeric_cols,
    outputCol="features_raw"
)

# Scale numerical features (optional but good practice)
scaler = StandardScaler(inputCol="features_raw", outputCol="features")

# Set up Logistic Regression model within a Pipeline in Spark ML

In [4]:
lr = LogisticRegression(featuresCol="features", labelCol="churn")

pipeline = Pipeline(stages=indexers + encoders + [assembler, scaler, lr])

# Split training data into train and test and train the model

In [5]:
train, test = training_data.randomSplit([0.8, 0.2], seed=42)
model = pipeline.fit(train)

# Predict customer churn using trained model

In [6]:
predictions = model.transform(test)

# Keep customer_id with predictions
result = predictions.select("customerid", "probability", "prediction", "churn")

result.show(truncate=False)

+----------+------------------------------------------+----------+-----+
|customerid|probability                               |prediction|churn|
+----------+------------------------------------------+----------+-----+
|0057-QBUQH|[1.0,0.0]                                 |0.0       |0    |
|0174-QRVVY|[1.0,0.0]                                 |0.0       |0    |
|0343-QLUZP|[1.0,0.0]                                 |0.0       |0    |
|0420-BWTPW|[0.8932986034369781,0.10670139656302191]  |0.0       |1    |
|0661-XEYAN|[0.03780631989901093,0.962193680100989]   |1.0       |1    |
|0746-JTRFU|[0.00541133780780283,0.9945886621921972]  |1.0       |1    |
|0946-FKYTX|[0.9999999999967153,3.284705840655988E-12]|0.0       |0    |
|1043-YCUTE|[1.0,0.0]                                 |0.0       |1    |
|1536-HBSWP|[0.7631408815151772,0.23685911848482277]  |0.0       |0    |
|1589-AGTLK|[0.03073995285862471,0.9692600471413753]  |1.0       |1    |
|1599-EAHXY|[1.0,0.0]                              

# Evaluate the model accuracy

In [7]:
# Extract probability and label
scoreAndLabels = predictions.select("probability", "churn") \
    .rdd.map(lambda row: (float(row.probability[1]), float(row.churn)))

metrics = BinaryClassificationMetrics(scoreAndLabels)

print("Accuracy: ", round(metrics.areaUnderROC*100 , 1), "%")

Accuracy:  84.8 %


# Save the model

In [8]:
path = "/Workspace/models/customer_churn/ml_model"
model.write().overwrite().save(path)
print("Churn Prediction model saved at: " + path)

Churn Prediction model saved at: /Workspace/models/customer_churn/ml_model
